In [ ]:
import torch
import os

# 1. Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: No GPU detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

# 2. Set up folder structure
os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/val", exist_ok=True)
os.makedirs("./models", exist_ok=True)

print("Folders created successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Create synthetic sample folders and images for testing
for class_name in ["control", "treated"]:
    os.makedirs(f"./data/train/{class_name}", exist_ok=True)
    os.makedirs(f"./data/val/{class_name}", exist_ok=True)
    for i in range(10):
        img_arr = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        Image.fromarray(img_arr).save(f"./data/train/{class_name}/sample_{i}.png")
        Image.fromarray(img_arr).save(f"./data/val/{class_name}/sample_{i}.png")

# Data transforms optimized for scientific imaging (handling arbitrary orientations)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets and create dataloaders
train_dataset = datasets.ImageFolder(root="./data/train", transform=train_transforms)
val_dataset = datasets.ImageFolder(root="./data/val", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"Classes detected: {train_dataset.class_to_idx}")
print(f"Training batches ready: {len(train_loader)}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Get number of classes from our dataset
num_classes = len(train_dataset.classes)

# Load pre-trained ResNet18 backbone
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modify the final fully connected layer to include hidden layers, Batch Normalization, and Dropout
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, num_classes)
)

# Move model to the configured device (CPU or GPU)
model = model.to(device)

print(f"Model successfully built and moved to {device}!")
print(f"Adapted classification head: Linear({num_ftrs} -> 256) -> BatchNorm1d -> ReLU -> Dropout(0.3) -> Linear(256 -> {num_classes})")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Get number of classes from our training dataset
num_classes = len(train_dataset.classes)

# Load pre-trained ResNet18 backbone
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Optimize architecture: progressive classification head with projection, normalization, and regularization
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, num_classes)
)

# Move optimized model to active device
model = model.to(device)

print("Optimized ResNet18 Architecture built successfully!")
print(f"New fully-connected head: Linear({num_ftrs} -> 256) -> BatchNorm1d -> ReLU -> Dropout(0.3) -> Linear(256 -> {num_classes})")

In [ ]:

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn

# 1. Loss function, Optimizer, and Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 2. Training and Validation Loops
EPOCHS = 5
best_val_loss = float('inf')

print("Starting training pipeline...\n")

for epoch in range(EPOCHS):
    # Training Phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    train_loss = running_loss / total_train
    train_acc = correct_train / total_train

    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    val_loss = val_loss / total_val
    val_acc = correct_val / total_val

    # Adjust learning rate based on validation loss performance
    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    # Save the best model checkpoint to the models directory
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "./models/best_scientific_model.pth")
        print("--> Best model checkpoint saved successfully!")

print("\nPipeline execution complete! You have successfully built and run an end-to-end PyTorch scientific image classification pipeline.")

Starting training pipeline...

Epoch [1/5] | Train Loss: 0.8486 | Train Acc: 40.00% | Val Loss: 0.7130 | Val Acc: 50.00%
--> Best model checkpoint saved successfully!
Epoch [2/5] | Train Loss: 0.5773 | Train Acc: 60.00% | Val Loss: 0.8372 | Val Acc: 50.00%
Epoch [3/5] | Train Loss: 0.8919 | Train Acc: 60.00% | Val Loss: 0.7485 | Val Acc: 50.00%
Epoch [4/5] | Train Loss: 0.9135 | Train Acc: 50.00% | Val Loss: 1.1285 | Val Acc: 50.00%
Epoch [5/5] | Train Loss: 0.6497 | Train Acc: 65.00% | Val Loss: 1.0122 | Val Acc: 50.00%

Pipeline execution complete! You have successfully built and run an end-to-end PyTorch scientific image classification pipeline.


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn

# 1. Loss function, Optimizer, and Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 2. Training and Validation Loops
EPOCHS = 5
best_val_loss = float('inf')

print("Starting training pipeline...\n")

for epoch in range(EPOCHS):
    # Training Phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    train_loss = running_loss / total_train
    train_acc = correct_train / total_train

    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    val_loss = val_loss / total_val
    val_acc = correct_val / total_val

    # Adjust learning rate based on validation loss performance
    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    # Save the best model checkpoint to the models directory
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "./models/best_scientific_model.pth")
        print("--> Best model checkpoint saved successfully!")

print("\nPipeline execution complete! You have successfully built and run an end-to-end PyTorch scientific image classification pipeline.")

Starting training pipeline...

Epoch [1/5] | Train Loss: 0.7554 | Train Acc: 55.00% | Val Loss: 0.7320 | Val Acc: 55.00%
--> Best model checkpoint saved successfully!
Epoch [2/5] | Train Loss: 1.1210 | Train Acc: 35.00% | Val Loss: 0.9089 | Val Acc: 50.00%
Epoch [3/5] | Train Loss: 0.7759 | Train Acc: 60.00% | Val Loss: 0.7405 | Val Acc: 50.00%
Epoch [4/5] | Train Loss: 0.8170 | Train Acc: 55.00% | Val Loss: 0.7460 | Val Acc: 50.00%
Epoch [5/5] | Train Loss: 0.7261 | Train Acc: 45.00% | Val Loss: 0.7429 | Val Acc: 50.00%

Pipeline execution complete! You have successfully built and run an end-to-end PyTorch scientific image classification pipeline.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# 1. Setup device and ensure model template exists
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate val_dataset and val_loader if missing
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

try:
    val_dataset = datasets.ImageFolder(root="./data/val", transform=val_transforms)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
except Exception:
    val_dataset = None
    val_loader = None

# Initialize model if not present
try:
    _ = model
except NameError:
    num_classes = len(val_dataset.classes) if val_dataset else 2
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_ftrs, num_classes)
    )

model = model.to(device)

# Load the best saved model checkpoint
try:
    model.load_state_dict(torch.load("./models/best_scientific_model.pth", map_location=device))
    model.eval()
    print("Model weights loaded successfully!")
except Exception as e:
    print(f"Warning loading checkpoint: {e}")

all_preds = []
all_labels = []

# 2. Run inference safely
if val_loader:
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    # 3. Generate and print classification metrics
    class_names = list(val_dataset.class_to_idx.keys())

    print("=== Classification Report ===")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    print("\n=== Confusion Matrix ===")
    print(confusion_matrix(all_labels, all_preds))
else:
    print("Validation loader is not available. Please ensure './data/val' directory contains test classes.")

Model weights loaded successfully!
=== Classification Report ===
              precision    recall  f1-score   support

     control       0.53      1.00      0.69        10
     treated       1.00      0.10      0.18        10

    accuracy                           0.55        20
   macro avg       0.76      0.55      0.44        20
weighted avg       0.76      0.55      0.44        20


=== Confusion Matrix ===
[[10  0]
 [ 9  1]]


In [ ]:
### Project Directory Structure

This is the layout of your scientific image classifier project:

```text
. (Project Root)
├── data/
│   ├── train/
│   │   ├── control/ (10 images)
│   │   └── treated/ (10 images)
│   └── val/
│       ├── control/ (10 images)
│       └── treated/ (10 images)
├── models/
│   └── best_scientific_model.pth
```

SyntaxError: invalid character '├' (U+251C) (2384993487.py, line 7)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn

# Re-initialize loss function, optimizer, and scheduler for the updated model
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 10
best_val_loss = float('inf')

print("Starting training loop with the updated model architecture...\n")

for epoch in range(EPOCHS):
    # Training Phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    train_loss = running_loss / total_train
    train_acc = correct_train / total_train

    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    val_loss = val_loss / total_val
    val_acc = correct_val / total_val

    # Step the learning rate scheduler based on validation loss
    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    # Save checkpoint if validation loss improves
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "./models/best_scientific_model.pth")
        print("--> New best model checkpoint saved!")

print("\nTraining complete!")

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

def main():
    set_seed(42)

    # Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Directories
    os.makedirs("./models", exist_ok=True)

    # Data augmentations optimized for scientific imagery
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Load datasets
    train_dataset = datasets.ImageFolder(root="./data/train", transform=train_transforms)
    val_dataset = datasets.ImageFolder(root="./data/val", transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    num_classes = len(train_dataset.classes)
    print(f"Classes detected: {train_dataset.class_to_idx}")

    # Model definition
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_features, num_classes)
    )
    model = model.to(device)

    # Loss, Optimizer, Scheduler
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    EPOCHS = 10
    best_val_loss = float('inf')

    print("\nStarting training pipeline...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        train_loss = running_loss / total_train
        train_acc = correct_train / total_train

        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()

        val_loss = val_loss / total_val
        val_acc = correct_val / total_val

        scheduler.step(val_loss)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "./models/best_scientific_model.pth")
            print("--> Best model checkpoint saved successfully!")

    print("\nTraining complete!")

if __name__ == "__main__":
    main()

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn as nn

def evaluate_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Evaluating on device: {device}")

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    try:
        val_dataset = datasets.ImageFolder(root="./data/val", transform=val_transforms)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
        num_classes = len(val_dataset.classes)
    except Exception as e:
        print(f"Warning loading dataset: {e}")
        val_dataset = None
        val_loader = None
        num_classes = 2

    # Rebuild model structure
    model = models.resnet18(weights=None)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_features, num_classes)
    )

    # Load weights safely
    try:
        model.load_state_dict(torch.load("./models/best_scientific_model.pth", map_location=device))
        print("Successfully loaded weights from checkpoint.")
    except Exception as e:
        print(f"Could not load checkpoint ({e}). Running on uninitialized weights.")

    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    if val_loader:
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())

        class_names = list(val_dataset.class_to_idx.keys())

        print("\n=== Classification Report ===")
        print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

        print("=== Confusion Matrix ===")
        print(confusion_matrix(all_labels, all_preds))
    else:
        print("Validation loader is not configured. Please ensure your './data/val' directory is populated.")

if __name__ == "__main__":
    evaluate_model()

In [ ]:
import os

train_dir = "./data/train"
classes = os.listdir(train_dir)

print("=== Training Set Class Distribution ===")
for cl in sorted(classes):
    class_path = os.path.join(train_dir, cl)
    if os.path.isdir(class_path):
        num_images = len([f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))])
        print(f"Class '{cl}': {num_images} images")

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

def main():
    set_seed(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    os.makedirs("./models", exist_ok=True)

    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(root="./data/train", transform=train_transforms)
    val_dataset = datasets.ImageFolder(root="./data/val", transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    num_classes = len(train_dataset.classes)
    print(f"Classes detected: {train_dataset.class_to_idx}")

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_features, num_classes)
    )
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    EPOCHS = 15
    best_val_loss = float('inf')

    print("\nStarting training pipeline...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        train_loss = running_loss / total_train
        train_acc = correct_train / total_train

        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()

        val_loss = val_loss / total_val
        val_acc = correct_val / total_val

        scheduler.step(val_loss)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "./models/best_scientific_model.pth")
            print("--> Best model checkpoint saved successfully!")

    print("\nTraining complete!")

if __name__ == "__main__":
    main()

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn as nn

def evaluate_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Evaluating on device: {device}")

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_dataset = datasets.ImageFolder(root="./data/val", transform=val_transforms)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    num_classes = len(val_dataset.classes)

    # Rebuild model structure
    model = models.resnet18(weights=None)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_features, num_classes)
    )

    # Load weights
    model.load_state_dict(torch.load("./models/best_scientific_model.pth", map_location=device))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    class_names = list(val_dataset.class_to_idx.keys())

    print("\n=== Classification Report ===")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    print("=== Confusion Matrix ===")
    print(confusion_matrix(all_labels, all_preds))

if __name__ == "__main__":
    evaluate_model()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix():
    # Generate predictions from evaluation
    # Using the same labels and predictions stored in the last evaluation
    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

plot_confusion_matrix()

### Analysis: Why is the model predicting only one class?

Based on your notebook execution outputs and kernel state, your model is collapsing to predict only a single class (predicting everything as `treated` or experiencing high confusion). Here are the main reasons why this is happening and how to address it:

#### 1. Synthetic Random Noise Data
* **The Issue:** In the mock dataset creation step (Cell `FIkCDWZo3any`), the training and validation images are generated using `np.random.randint(0, 255, (224, 224, 3))`.
* **The Impact:** Pure random noise contains no learnable scientific features, patterns, or edges. Because there is no real underlying signal distinguishing `control` from `treated`, the model cannot find a meaningful decision boundary. As a result, it defaults to the easiest mathematical shortcut to reduce training loss, which often means predicting the majority class or a single class arbitrary bias.

#### 2. Extremely Small Dataset Size
* **The Issue:** There are only 10 training samples and 10 validation samples per class (20 total images per dataset split).
* **The Impact:** Deep networks like ResNet18 contain millions of parameters. When trained on only 20 samples of random noise, the model instantly overfits or fails to converge, causing the classifier head weight updates to become highly unstable and collapse into single-class outputs.

#### 3. High Learning Rate on Small Data
* **The Issue:** The initial optimization attempt used a learning rate of `0.001` with `AdamW`.
* **The Impact:** On very small datasets, a high learning rate causes the gradients to oscillate violently, quickly destroying pre-trained feature extractors and causing the weights of the final classification layer to diverge or saturate towards one extreme bias.

---

### How to Fix This:
1. **Replace Synthetic Data with Real Images:** Load your actual scientific dataset. Real structured patterns will give the CNN features (like textures, cell structures, or color variations) to learn.
2. **Increase Dataset Volume:** Aim for at least 100+ images per class to prevent overfitting.
3. **Fine-tune Conservatively:** Keep the learning rate low (e.g., `1e-4` or `5e-5`) when training on custom scientific imagery, and continue using pre-trained weights to leverage robust feature representation.